# Kernel Level Profiling

We start by compiling and executing our application to make sure that its results still match our expectations.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v1.cpp -o ../build/stencil-2d-omp-target-v1

In [ ]:
!../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

## Nsight Compute CLI

Next, we profile the application using the CLI of Nsight Compute: `ncu`.
Command line arguments are:
* `-o ...`: sets the target output profile file (equivalent to `nsys`)
* `--force-overwrite`: replaces the profile file if it already exists (in contrast to `nsys` no `=true`)

All command line arguments are listed in the [documentation](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html#profile).

Without an output file, results are printed to the command line.

Note that we also decrease the number of iterations since by default *every kernel* is profiled.

In [ ]:
!ncu ../build/stencil-2d-omp-target-v1 double 8192 8192 2 2

We can further limit the scope of profiled kernels with
* `--launch-skip n` or `-s n`: skips the first `n` kernels encountered
* `--launch-count n` or `-c n`: limits profiling to the first `n` applicable kernels
* `--kernel name` or `-k name`: limits profiling to kernels with the name `name`
  * can also be used with regex, e.g. `-k regex:"stencil[1-3]d"`
  * Nsight Compute also supports kernel renaming

In [ ]:
!ncu -s 2 -c 1 ../build/stencil-2d-omp-target-v1 double 8192 8192 2 2

### Potential Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_6 (64, 1, 1)x(128, 1, 1), Context 1, Stream 13, Device 0, CC 8.6
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         7.24
    SM Frequency                    Ghz         1.30
    Elapsed Cycles                cycle    9,325,096
    Memory Throughput                 %        42.95
    DRAM Throughput                   %        21.73
    Duration                         ms         7.15
    L1/TEX Cache Throughput           %        58.04
    L2 Cache Throughput               %        35.86
    SM Active Cycles              cycle 6,899,639.96
    Compute (SM) Throughput           %        17.13
    ----------------------- ----------- ------------

    OPT   This kernel grid is too small to fill the available resources on this device, resulting in only 0.06 full     
          waves across all SMs. Look at Launch Statistics for more details.                                             
```

```
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   128
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                     64
    Registers Per Thread             register/thread              36
    Shared Memory Configuration Size           Kbyte           16.38
    Driver Shared Memory Per Block       Kbyte/block            1.02
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    # SMs                                         SM              84
    Stack Size                                                 1,024
    Threads                                   thread           8,192
    # TPCs                                                        42
    Enabled TPC IDs                                              all
    Uses Green Context                                             0
    Waves Per SM                                                0.06
    -------------------------------- --------------- ---------------

    OPT   Est. Speedup: 23.81%                                                                                          
          The grid for this launch is configured to execute only 64 blocks, which is less than the 84 multiprocessors   
          used. This can underutilize some multiprocessors. If you do not intend to execute this kernel concurrently    
          with other workloads, consider reducing the block size to have at least one block per multiprocessor or       
          increase the size of the grid to fully utilize the available hardware resources. See the Hardware Model       
          (https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-hw-model) description for more      
          details on launch configurations.                                                                             
```

```
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           16
    Block Limit Registers                 block           12
    Block Limit Shared Mem                block           16
    Block Limit Warps                     block           12
    Theoretical Active Warps per SM        warp           48
    Theoretical Occupancy                     %          100
    Achieved Occupancy                        %         8.33
    Achieved Active Warps Per SM           warp         4.00
    ------------------------------- ----------- ------------

    OPT   Est. Local Speedup: 91.67%                                                                                    
          The difference between calculated theoretical (100.0%) and measured achieved occupancy (8.3%) can be the      
          result of warp scheduling overheads or workload imbalances during the kernel execution. Load imbalances can   
          occur between warps within a block as well as across blocks of the same kernel. See the CUDA Best Practices   
          Guide (https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#occupancy) for more details on     
          optimizing occupancy.                                                                                         
```

```
    Section: GPU and Memory Workload Distribution
    -------------------------- ----------- -------------
    Metric Name                Metric Unit  Metric Value
    -------------------------- ----------- -------------
    Average DRAM Active Cycles       cycle 11,248,417.33
    Total DRAM Elapsed Cycles        cycle   621,136,896
    Average L1 Active Cycles         cycle  6,899,639.96
    Total L1 Elapsed Cycles          cycle   783,173,268
    Average L2 Active Cycles         cycle  8,759,037.29
    Total L2 Elapsed Cycles          cycle   421,879,584
    Average SM Active Cycles         cycle  6,899,639.96
    Total SM Elapsed Cycles          cycle   783,173,268
    Average SMSP Active Cycles       cycle     6,895,059
    Total SMSP Elapsed Cycles        cycle 3,132,693,072
    -------------------------- ----------- -------------

    OPT   Est. Speedup: 19.24%                                                                                          
          One or more SMs have a much lower number of active cycles than the average number of active cycles. Maximum   
          instance value is 26.00% above the average, while the minimum instance value is 100.00% below the average.    
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 19.19%                                                                                          
          One or more SMSPs have a much lower number of active cycles than the average number of active cycles. Maximum 
          instance value is 25.95% above the average, while the minimum instance value is 100.00% below the average.    
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 19.24%                                                                                          
          One or more L1 Slices have a much lower number of active cycles than the average number of active cycles.     
          Maximum instance value is 26.00% above the average, while the minimum instance value is 100.00% below the     
          average.                                                                                                      
```

## Exercise - Interpret Compute Output

Have a look at the speed of light (SOL) output and try to pinpoint performance issues, as well as their main contributor.

### Possible Solution

Nsight Compute rightfully reports that none of the usual performance limiters (FLOP/s, MEM, ...) are fully utilized which hints at an underperforming application.
The tool also tries to give developers hints into potentially useful optimizations or reasons for lacking performance.
In this case these hints are useful (which might not always be the case):
* The occupancy is very low which is a result of having only one fraction of a wave.
* This stems from a low number of CUDA blocks, lower than the number of SMs even.

## Next Step

To better understand the architecture of GPUs, and what we need to fully utilize their potential, head to the [GPU architecture](./05-gpu-architecture.ipynb) notebook.